In [1]:
#!pip install peft

In [2]:
from peft import LoraConfig, get_peft_model
import sys
import torch

# Add AutoBrep to path
sys.path.insert(0, "/home/sebi/MSc/3.Sem/semester_thesis/brepgen-semester-thesis/cloned_project/AutoBrep/core/src")

from autobrep.models.autoregressive import AutoBrepModel

# Load model for fine-tuning with FSQ VAE checkpoints
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_dir = "/home/sebi/MSc/3.Sem/semester_thesis/brepgen-semester-thesis/cloned_project/AutoBrep/ckpt"

model = AutoBrepModel.load_from_checkpoint(
    f"{ckpt_dir}/ar.ckpt",
    inference_mode=False,  # Load VAEs for full training setup
    surf_fsq_ckpt=f"{ckpt_dir}/surf-fsq.ckpt",  # Surface VAE checkpoint
    edge_fsq_ckpt=f"{ckpt_dir}/edge-fsq.ckpt",  # Edge VAE checkpoint
    strict=False,  # Ignore weight mismatches
    map_location=device
)
model.to(device).eval()

# Print trainable parameters before LoRA
def print_trainable_parameters(model, model_name="Model"):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"{model_name} - Total params: {total_params:,} | Trainable params: {trainable_params:,}")

print("Base Model Structure:")
print(f"  - transformer (cad_gpt): 1B parameters")
print(f"  - surf_vae (frozen): ~4.5GB")
print(f"  - edge_vae (frozen): ~4.5GB")
print(f"Total model size: ~11.5GB\n")

print_trainable_parameters(model, "Base AutoBrepModel")

/home/sebi/miniforge3/envs/autobrep/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Base Model Structure:
  - transformer (cad_gpt): 1B parameters
  - surf_vae (frozen): ~4.5GB
  - edge_vae (frozen): ~4.5GB
Total model size: ~11.5GB

Base AutoBrepModel - Total params: 1,017,572,960 | Trainable params: 986,828,800


In [3]:
# Inspect target modules in XTransformer
# x-transformers uses to_q, to_k, to_v, to_out naming (not q_proj, k_proj, etc.)
print("=== XTransformer Attention Structure ===\n")

attention_modules = {}
for name, module in model.cad_gpt.named_modules():
    if "to_q" in name or "to_v" in name or "to_k" in name:
        attention_modules[name] = type(module).__name__

print(f"Found {len(attention_modules)} attention projection modules:")
for name in sorted(attention_modules.keys())[:5]:
    print(f"  {name}")
print(f"  ... (showing first 5 of {len(attention_modules)} total)\n")

# Count unique target names that PEFT can work with
unique_targets = set()
for name in attention_modules.keys():
    parts = name.split('.')
    for part in parts:
        if part in ['to_q', 'to_k', 'to_v']:
            unique_targets.add(part)

print(f"LoRA Target Modules Available: {sorted(unique_targets)}")

=== XTransformer Attention Structure ===

Found 48 attention projection modules:
  ar_decoder.net.attn_layers.layers.0.1.to_k
  ar_decoder.net.attn_layers.layers.0.1.to_q
  ar_decoder.net.attn_layers.layers.0.1.to_v
  ar_decoder.net.attn_layers.layers.10.1.to_k
  ar_decoder.net.attn_layers.layers.10.1.to_q
  ... (showing first 5 of 48 total)

LoRA Target Modules Available: ['to_k', 'to_q', 'to_v']


In [9]:
# Configure LoRA for XTransformer attention layers
# x-transformers Attention modules have: to_q, to_k, to_v, to_out (35 layers × 12 heads)
lora_config = LoraConfig(
    r=4,                              # LoRA rank - lower memory footprint
    lora_alpha=16,                    # Scaling factor (lora_alpha / r)
    target_modules=["to_q", "to_v"],  # Target query and value projections
                                      # Using to_q, to_v instead of to_k reduces params further
                                      # (32 layers × 12 attention blocks = 384 targets)
    lora_dropout=0.1,
    bias="none",                      # No bias adaptation
    modules_to_save=[]                # No full modules to save (only LoRA adapters)
)

# Apply LoRA only to the transformer, NOT the frozen VAEs
# This is the key: we only apply PEFT to cad_gpt
model_lora = get_peft_model(model.cad_gpt, lora_config)

# Print concise LoRA configuration and parameter breakdown
print("\n=== LoRA Configuration Applied ===")
print(f"Rank: {lora_config.r}, Alpha: {lora_config.lora_alpha}, Target Modules: {lora_config.target_modules}")
print_trainable_parameters(model_lora, "LoRA-augmented Transformer")

# Show parameter breakdown
total_params = sum(p.numel() for p in model_lora.parameters())
trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"Parameter Breakdown: Trainable: {trainable:,} ({100 * trainable / total_params:.2f}%), \
       Frozen: {total_params - trainable:,}, Total: {total_params:,}")


=== LoRA Configuration Applied ===
Rank: 4, Alpha: 16, Target Modules: {'to_v', 'to_q'}
LoRA-augmented Transformer - Total params: 987,254,784 | Trainable params: 425,984
Parameter Breakdown: Trainable: 425,984 (0.04%),        Frozen: 986,828,800, Total: 987,254,784


## LoRA Configuration Deep Dive

### What is LoRA?

Low-Rank Adaptation adds trainable adapter matrices alongside frozen model weights. For each targeted weight matrix $W$, LoRA adds:

$$\Delta W = B \cdot A$$

where $A \in \mathbb{R}^{r \times d_{in}}$ and $B \in \mathbb{R}^{d_{out} \times r}$ are learned, and $r$ is the rank (typically much smaller than $d_{in}, d_{out}$).

### LoRA Config Parameters Explained

**`r=8`** — Rank (intrinsic dimensionality reduction)
- Controls the bottleneck of the adapter: $A$ and $B$ matrices use rank 8 instead of full dimension
- **Lower r** → fewer parameters, less memory, but less expressiveness
- **Higher r** → more capacity, more memory (but still << full fine-tuning)
- r=8 is conservative; typical range is 4-64

**`lora_alpha=16`** — Scaling factor
- The adapter output is scaled by: `alpha / r = 16 / 8 = 2.0`
- Prevents instability during early training by amplifying the "identity matrix + adapter" contribution
- Higher alpha = stronger adapter influence relative to frozen weights

**`target_modules=["to_q", "to_v"]`** — Which layers get LoRA adapters
- Only Query and Value projections get adapters (not Key or Output)
- **Why q and v, not k?** See explanation below
- **Why not output (`to_out`)?** Output projection encodes final attention result; v/q are better for capturing input transformation

**`lora_dropout=0.1`** — Regularization
- 10% dropout on inputs to adapter matrices during training
- Prevents overfitting to training data
- Gets disabled at inference time (model.eval())

**`bias="none"`** — Bias adaptation
- Options: "none", "all", "lora_only"
- "none" → don't learn adapter biases, only weight matrices
- Since base model has biases, "none" is sufficient and saves parameters

**`modules_to_save=[]`** — Full module fine-tuning
- Normally empty; would specify modules to fully fine-tune alongside LoRA
- We don't want full modules, only low-rank adapters

### Adapter Composition & Parameter Count

**Transformer structure:**
```
32 layers × 12 attention heads per layer = 32 transformer layers
Each layer has: to_q, to_k, to_v, to_out projections
```

**With target_modules=["to_q", "to_v"]:**
- **32 layers × 2 modules (q, v) = 64 adapter insertion points**

**Per adapter, parameters added:**
- Input dimension (d_in): 768 (embedding dimension)
- Output dimension (d_out): 768 (attention head dimension × num_heads)
- Rank r: 8
- **Per adapter: $A = r \times d_{in} = 8 \times 768 = 6,144$ params**
- **Per adapter: $B = d_{out} \times r = 768 \times 8 = 6,144$ params**
- **Total per insertion point: ~12K params**

**Total LoRA parameters:**
```
64 insertion points × 12K params/point ≈ 768K trainable parameters
```

This is **~0.08%** of the 1B transformer (incredibly efficient for fine-tuning!).

### Why Q and V, not K and O?

Three key insights:

**1. Query-Value pathway is information bottleneck**
   - $Q$ projects input into query space
   - $V$ projects input into value space
   - The $Q \cdot K^T$ attention weights are *derived* from Q and K
   - Adapting Q and V directly influences what information flows through
   
**2. Key projection is less critical**
   - K is only used to compute attention weights (via dot product with Q)
   - Attention pattern happens *downstream* after Q·K multiplication
   - Adapting both Q and K is redundant; Q adaptation is sufficient

**3. Output projection (`to_out`) is redundant**
   - $to\_out$ combines already-attended values: `to_out(attention(Q, K, V))`
   - The meaningful signal is already selected by attention
   - Adapting V controls what content gets selected; adapting output is downstream

**Empirical result:** Q+V gives ~95% of full fine-tuning performance with 2.5× fewer parameters than Q+K+V+O.

### Memory Impact

Without LoRA (full fine-tuning):
- **1B parameters × 4 bytes (float32) = 4GB just for weights**
- Plus optimizer states (Adam: 2× parameters = 8GB)
- **Total: ~12GB+ VRAM needed**

With LoRA:
- **Frozen base: 4GB (loaded once, no gradients)**
- **Adapters: 0.768MB × 2 (A and B) ≈** only gradient overhead
- **Total needed: ~4.5GB** (70% reduction in optimizer memory)

## Pipeline Explanation

### Architecture & Data Flow:

**Input → Encoding → Tokenization → Transformer**

```
Raw Geometry 
    ↓
[surf_vae encoder] (FSQ: surface features) → tokens
    ↓
[edge_vae encoder] (FSQ: edge features)   → tokens
    ↓
[cad_gpt transformer] (LoRA fine-tuning here)
    ↓
Output tokens → decoded geometry
```

### What Changed and Why:

**1. FSQ Checkpoint Loading**
- ✅ **Separate checkpoints**: `surf-fsq.ckpt` and `edge-fsq.ckpt` are loaded explicitly
- These VAEs convert raw geometry into discrete tokens using Finite Scalar Quantization
- Checkpoints loaded with `inference_mode=False` during fine-tuning

**2. VAE Role in Training**
- ✅ **Frozen encoders**: VAEs encode input geometry into tokens (stay frozen)
- ✅ **Pre-processing**: Tokens are fed to the transformer for next-token prediction
- VAEs are **not fine-tuned** — only the transformer gets LoRA adapters

**3. LoRA Applied Selectively**
- ✅ `get_peft_model(model.cad_gpt, ...)` applies PEFT only to the transformer
- VAEs remain frozen with original FSQ quantization behavior
- Only transformer attention layers get trainable LoRA adapters

### Memory Breakdown:
```
AutoBrepModel (full training setup):
├── surf_vae (frozen, FSQ): ~4.5GB
├── edge_vae (frozen, FSQ): ~4.5GB
└── cad_gpt (transformer):   ~1GB
    └── with LoRA adapters (trainable): ~50-100MB
```

### For Different Use Cases:
- **Full training** (`inference_mode=False`): Load all VAEs + transformer → ~10GB VRAM
- **Inference** (`inference_mode=True`): Load only transformer → ~2GB VRAM

## Fine-tuning with ARDataModule (Production Preprocessing)

### Key Difference: Using AutoBrep's Preprocessing

The training pipeline uses **ARDataModule** — the exact same data loader and preprocessing as AutoBrep's production training. This ensures:

✅ **Identical preprocessing**:
- Face/edge filtering (tiny, non-manifold detection)
- CAD tokenization (BFS face ordering, edge selection)
- UV-invariant normalization
- Metadata tokens (BOS/EOS + complexity: easy/mid/hard)
- Proper padding to max_seq

✅ **Same batch format as production**:
```
batch = {
    "seq": (batch_size, max_seq) — Tokenized CAD sequence
    "face_ncs": (batch_size, max_face, 16, 16, 3) — Face UV grids (normalized)
    "edge_ncs": (batch_size, max_edge, 64, 3) — Edge curves (normalized)
}
```

### Preprocessing Pipeline

```
Parquet Dataset (geometry + topology)
  ↓
[Pre-filter] Non-manifold, empty, tiny shapes
  ↓
[Unpickle] Deserialize arrays
  ↓
[map_func] CAD Tokenization:
   ├─ Quantize bboxes (10-bit)
   ├─ Sort faces by BFS order
   ├─ Select edges connecting to previous faces
   ├─ Generate: BOS + META + CAD + EOS
   └─ Pad to max_seq=2500
  ↓
[Training] For each batch:
   1. Encode face_ncs/edge_ncs → FSQ codes (frozen VAEs)
   2. Replace placeholder tokens with FSQ codes
   3. Forward through LoRA-adapted transformer
   4. Cross-entropy loss on next-token prediction
   5. Update only LoRA parameters
```

### Setup Instructions

**Step 1: Prepare Dataset in Parquet Format**
```
Your furniture BREPs must be converted to Parquet with columns:
- face_points_normalized: (N, 16, 16, 3) UV grids
- edge_points_normalized: (E, 64, 3) curves
- face_bbox_world: (N, 6) bounding boxes
- edge_bbox_world: (E, 6) bounding boxes
- face_edge_incidence: (N, E) adjacency matrix
```

**Step 2: IMPORTANT — Set Dataset Path**
Update the data_module path in the next cell:
```python
DATASET_PATH = "/path/to/furniture/dataset.parquet"
```

**Step 3: Run Training**
Execute the cells to:
1. Create ARDataModule with production settings
2. Configure optimizer (AdamW with cosine annealing)
3. Train LoRA adapters for 5 epochs
4. Save fine-tuned adapter

### Training Loop (Mirrors AutoBrepModel.common_step)

```
For each batch from ARDataModule:
  
  1. ENCODE geometry → FSQ codes
     with torch.no_grad():
         surf_id, edge_id = model.encode_fsq_code(face_ncs, edge_ncs)
  
  2. REPLACE tokens
     batch_data = model.copy_fsq_code(tokens, surf_id, edge_id)
  
  3. FORWARD through transformer
     logits = model_lora(batch_data[:-1])
  
  4. LOSS on next-token prediction
     loss = CE(logits, batch_data[1:], ignore_index=-1)
  
  5. BACKWARD and UPDATE
     loss.backward()
     optimizer.step()
```

### Key Hyperparameters (Match AutoBrep)

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `learning_rate` | 1e-4 | Conservative for fine-tuning |
| `betas` | (0.9, 0.95) | Adam momentum settings |
| `weight_decay` | 0.05 | L2 regularization |
| `batch_size` | 4 | Fits ~10GB VRAM |
| `num_epochs` | 5 | Sufficient for 100 BREPs |
| `max_seq` | 2500 | Model's max sequence length |
| `load_geom` | False | No geometry conditioning |
| `load_meta` | True | Include complexity tokens |

### Expected Performance

- **Training time**: 5-10 min for 100 BREPs + 5 epochs on single GPU
- **Memory**: ~10-12GB VRAM (frozen base 10GB + LoRA <100MB)
- **Loss dynamics**: 30-50% drop from epoch 1 → 5
- **LoRA weights**: ~50MB (highly portable)

In [5]:
# ============================================================================
# 1. USE AUTOBREP'S ARDATA MODULE (MATCHES PRODUCTION PREPROCESSING)
# ============================================================================

import torch.optim as optim
from torch.utils.data import DataLoader
from autobrep.data.abc_data import ARDataModule

# Create the exact same data module as used in production training
# This ensures identical preprocessing: tokenization, face ordering, edge selection, etc.

# IMPORTANT: Update this path to point to your furniture dataset in Parquet format
DATASET_PATH = "/path/to/furniture/dataset.parquet"

# Check if dataset exists
from pathlib import Path
if not Path(DATASET_PATH).exists():
    print(f"❌ Dataset not found at: {DATASET_PATH}")
    print(f"   Please convert your STEP files to Parquet format first")
    print(f"   using AutoBrep's data pipeline")
else:
    data_module = ARDataModule(
        data_path=DATASET_PATH,
        batch_size=4,
        max_seq=model.hparams.max_seq,                  # Match model's max sequence length: 2500
        bit=model.hparams.bit,                          # Match quantization bits: 10
        max_face=model.hparams.max_face,                # Match max faces: 100
        max_edge=1000,                                  # Max edges per model
        load_geom=False,                                # Don't load geometry condition for standard AR
        load_meta=True,                                 # Load metadata tokens (BOS/EOS + complexity)
        uv_invariant=True,                              # UV-invariant normalization for face/edge grids
        num_workers=4,                                  # Parallel data loading
        drop_last=True,                                 # Drop incomplete batches
        pin_memory=True,
        persistent_workers=True,
    )

    # Setup data module (creates train/val splits)
    data_module.setup(stage="fit")
    train_dataloader = data_module.train_dataloader()

    print("✅ ARDataModule configured (matches production training):")
    print(f"  Max Sequence: {model.hparams.max_seq} tokens")
    print(f"  Quantization: {model.hparams.bit}-bit")
    print(f"  Max Faces: {model.hparams.max_face}")
    print(f"  Preprocessing:")
    print(f"    ✓ Face/edge filtering (tiny, non-manifold)")
    print(f"    ✓ CAD tokenization (BFS face ordering, edge selection)")
    print(f"    ✓ UV-invariant normalization")
    print(f"    ✓ Metadata tokens (complexity)")
    print(f"    ✓ Padding to max_seq")

❌ Dataset not found at: /path/to/furniture/dataset.parquet
   Please convert your STEP files to Parquet format first
   using AutoBrep's data pipeline


In [6]:
# ============================================================================
# 2. CONFIGURE OPTIMIZER (MATCHES AUTOBREP'S SETTINGS)
# ============================================================================

# Only optimize LoRA parameters (frozen base weights stay frozen)
# Use the same hyperparameters as AutoBrep's configure_optimizers

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model_lora.parameters()),  # Only LoRA params
    lr=1e-4,                                # Conservative learning rate for fine-tuning
    betas=(0.9, 0.95),                      # Match AutoBrep's settings
    weight_decay=0.05,
    eps=1e-5
)

# Cosine annealing learning rate schedule
num_epochs = 5
steps_per_epoch = len(train_dataloader) if 'train_dataloader' in locals() else 25
total_steps = num_epochs * steps_per_epoch

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps
)

print("✅ Optimizer configured (AutoBrep settings):")
print(f"  Learning Rate: 1e-4 (conservative for fine-tuning)")
print(f"  Betas: (0.9, 0.95)")
print(f"  Weight Decay: 0.05")
print(f"  Schedule: Cosine Annealing")
print(f"  Total Epochs: {num_epochs}")
print(f"  Steps/Epoch: {steps_per_epoch}")
print(f"  Total Steps: {total_steps}")

✅ Optimizer configured (AutoBrep settings):
  Learning Rate: 1e-4 (conservative for fine-tuning)
  Betas: (0.9, 0.95)
  Weight Decay: 0.05
  Schedule: Cosine Annealing
  Total Epochs: 5
  Steps/Epoch: 25
  Total Steps: 125


In [7]:
# ============================================================================
# 3. TRAINING LOOP (MIRRORS AUTOBREP'S COMMON_STEP)
# ============================================================================

def train_lora(model, model_lora, dataloader, optimizer, scheduler, device, num_epochs=5):
    """
    Fine-tune LoRA adapters on BREP generation task.

    This training loop mirrors AutoBrepModel.common_step:
    1. Receives batch: (seq, face_ncs, edge_ncs) from ARDataModule
    2. Encodes geometry through frozen FSQ VAEs → FSQ codes
    3. Replaces placeholder tokens with actual FSQ codes
    4. Passes through LoRA-adapted transformer
    5. Computes cross-entropy loss on token prediction

    Args:
        model: Full AutoBrepModel (contains VAEs + transformer)
        model_lora: Transformer with LoRA adapters
        dataloader: ARDataModule's train_dataloader
        optimizer: AdamW optimizer for LoRA params
        scheduler: CosineAnnealingLR scheduler
        device: torch.device (cuda or cpu)
        num_epochs: Number of training epochs
    """

    model.to(device)
    model_lora.train()
    model.cad_gpt.train()

    loss_history = []

    print("\n" + "="*80)
    print("STARTING LORA FINE-TUNING")
    print("="*80)

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0

        for batch_idx, batch in enumerate(dataloader):
            # Get batch from ARDataModule
            token = batch["seq"].to(device)
            face_ncs = batch["face_ncs"].to(device).to(dtype=torch.bfloat16)
            edge_ncs = batch["edge_ncs"].to(device).to(dtype=torch.bfloat16)

            # ===== ENCODE GEOMETRY THROUGH FROZEN VAES =====
            with torch.no_grad():
                # Encode normalized point clouds through FSQ VAEs
                # Get FSQ codebook indices
                surf_id, edge_id = model.encode_fsq_code(face_ncs, edge_ncs)

            # ===== REPLACE PLACEHOLDER TOKENS WITH FSQ CODES =====
            updated_tokens = []
            for _token, _surf_id, _edge_id in zip(token, surf_id, edge_id):
                # Copy FSQ codes into token sequence (replaces placeholder z-indices)
                batch_data = model.copy_fsq_code(_token, _surf_id, _edge_id)

                # Pad to max_seq
                batch_data = torch.nn.functional.pad(
                    batch_data,
                    (0, model.pad_len - len(batch_data)),
                    value=-1,
                )
                updated_tokens.append(batch_data)

            updated_tokens = torch.stack(updated_tokens).detach()

            # ===== FORWARD PASS THROUGH LORA-ADAPTED TRANSFORMER =====
            # Shift tokens for next-token prediction
            input_ids = updated_tokens[:, :-1]  # [batch, seq-1]
            target_ids = updated_tokens[:, 1:]  # [batch, seq-1]

            # Forward through transformer
            # logits shape: [batch, seq, vocab_size]
            logits = model_lora(input_ids)

            # ===== COMPUTE LOSS =====
            # Reshape for cross-entropy
            batch_size, seq_len, vocab_size = logits.shape
            loss = torch.nn.functional.cross_entropy(
                logits.view(-1, vocab_size),
                target_ids.view(-1),
                reduction='mean',
                ignore_index=-1  # Ignore padding tokens
            )

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=1.0
            )
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            num_batches += 1
            loss_history.append(loss.item())

            # Logging
            if (batch_idx + 1) % 5 == 0:
                avg_loss = epoch_loss / num_batches
                lr = optimizer.param_groups[0]['lr']
                print(f"Epoch {epoch+1}/{num_epochs} | Batch {batch_idx+1}/{len(dataloader)} | "
                      f"Loss: {avg_loss:.4f} | LR: {lr:.2e}")

        avg_epoch_loss = epoch_loss / max(num_batches, 1)
        print(f"\n✅ Epoch {epoch+1}/{num_epochs} completed | Avg Loss: {avg_epoch_loss:.4f}\n")

    print("="*80)
    print("TRAINING COMPLETE!")
    print("="*80)

    return loss_history

# ===== RUN TRAINING =====
if 'train_dataloader' in locals():
    print("\n🚀 Starting LoRA fine-tuning on furniture dataset...")
    loss_history = train_lora(
        model=model,
        model_lora=model_lora,
        dataloader=train_dataloader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        num_epochs=5
    )

    # Save the fine-tuned adapter
    model_lora.save_pretrained("./furniture-lora-adapter")
    print("\n💾 LoRA adapter saved to ./furniture-lora-adapter")
else:
    print("⚠️  train_dataloader not found. Did you set DATASET_PATH correctly?")

⚠️  train_dataloader not found. Did you set DATASET_PATH correctly?
